In [3]:
# Schema overview understanding the 6 source tables
row_counts = spark.sql("""
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM orders
    UNION ALL
    SELECT 'order_products__prior', COUNT(*) FROM order_products__prior
    UNION ALL
    SELECT 'order_products__train', COUNT(*) FROM order_products__train
    UNION ALL
    SELECT 'products', COUNT(*) FROM products
    UNION ALL
    SELECT 'aisles', COUNT(*) FROM aisles
    UNION ALL
    SELECT 'departments', COUNT(*) FROM departments
""").toPandas()
row_counts.style \
    .format({'row_count': '{:,}'}) \
    .hide(axis='index')

StatementMeta(, 601bc22b-1501-49bd-ad74-e71b49112a96, 5, Finished, Available, Finished, False)

table_name,row_count
orders,"3,421,083"
order_products__prior,"32,434,489"
order_products__train,"1,384,617"
products,"49,688"
aisles,134
departments,21


In [4]:
# order_products — combine prior + train into one table since for Analytics 
spark.sql("""
    CREATE TABLE order_products AS
    SELECT *
    FROM order_products__prior
    UNION ALL
    SELECT *
    FROM order_products__train
""")

StatementMeta(, 601bc22b-1501-49bd-ad74-e71b49112a96, 6, Finished, Available, Finished, False)

DataFrame[]

In [5]:
# Orders
spark.sql("""
CREATE OR REPLACE TABLE clean_orders AS
SELECT
    order_id,
    user_id,
    eval_set,
    order_number,
    order_dow AS order_day_of_week,
    order_hour_of_day AS order_hour,
    days_since_prior_order
FROM orders
""")
spark.sql("DROP TABLE orders")
spark.sql("ALTER TABLE clean_orders RENAME TO orders")

spark.sql("DROP TABLE order_products__prior")
spark.sql("DROP TABLE order_products__train")

StatementMeta(, 601bc22b-1501-49bd-ad74-e71b49112a96, 7, Finished, Available, Finished, False)

DataFrame[]

In [7]:
# Schema overview after append tables
row_counts = spark.sql("""
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM orders
    UNION ALL
    SELECT 'order_products', COUNT(*) FROM order_products
    UNION ALL
    SELECT 'products', COUNT(*) FROM products
    UNION ALL
    SELECT 'aisles', COUNT(*) FROM aisles
    UNION ALL
    SELECT 'departments', COUNT(*) FROM departments
""").toPandas()
row_counts.style \
    .format({'row_count': '{:,}'}) \
    .hide(axis='index')

StatementMeta(, 601bc22b-1501-49bd-ad74-e71b49112a96, 9, Finished, Available, Finished, False)

table_name,row_count
orders,"3,421,083"
order_products,"33,819,106"
products,"49,688"
aisles,134
departments,21


In [12]:
# counts of unique users & orders 
validation = spark.sql("""
    SELECT
        'orders' AS table_name,
        COUNT(*) AS row_count,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT order_id) AS unique_orders
    FROM orders
    UNION ALL
    SELECT
        'order_products',
        COUNT(*),
        NULL,
        COUNT(DISTINCT order_id)
    FROM order_products

""").toPandas()

validation.style \
    .format({
        'row_count': '{:,}',
        'unique_users': '{:,}',
        'unique_orders': '{:,}'
    }) \
    .hide(axis='index')

StatementMeta(, 601bc22b-1501-49bd-ad74-e71b49112a96, 14, Finished, Available, Finished, False)

table_name,row_count,unique_users,unique_orders
orders,"3,421,083","206,209.0","3,421,083"
order_products,"33,819,106",nan,"3,346,083"


In [1]:
# orders table overview
Orders_KPIs = spark.sql("""
    SELECT
        COUNT(DISTINCT user_id) AS total_customers,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(COUNT(*) * 1.0
              / COUNT(DISTINCT user_id), 1) AS avg_orders_per_customer,
        ROUND(AVG(days_since_prior_order), 1) AS avg_days_between_orders,
        MIN(order_number) AS min_orders_Number,
        MAX(order_number) AS max_orders_Number
    FROM orders
""").toPandas()

Orders_KPIs_display = Orders_KPIs.T.reset_index()
Orders_KPIs_display.columns = ['KPI', 'Value']
Orders_KPIs_display.style.hide(axis='index')


StatementMeta(, b0aac211-79e1-4953-9624-01a6ec1e0cc7, 3, Finished, Available, Finished, False)

KPI,Value
total_customers,206209
total_orders,3421083
avg_orders_per_customer,16.6
avg_days_between_orders,11.100000
min_orders_Number,1
max_orders_Number,100


In [22]:
# products overview 
product_kpis = spark.sql("""
    SELECT
        COUNT(DISTINCT p.product_id) AS total_products,
        COUNT(DISTINCT a.aisle_id) AS total_aisles,
        COUNT(DISTINCT d.department_id) AS total_departments
    FROM products p
    LEFT JOIN departments d
      ON p.department_id = d.department_id
    LEFT JOIN aisles a  
      ON p.aisle_id = a.aisle_id  
""").toPandas()

product_kpis_display = product_kpis.T.reset_index()
product_kpis_display.columns = ['KPI', 'Value']
product_kpis_display.style.hide(axis='index')

StatementMeta(, 601bc22b-1501-49bd-ad74-e71b49112a96, 24, Finished, Available, Finished, False)

KPI,Value
total_products,49688
total_aisles,134
total_departments,21
